In [ ]:
conda create -n myenv python=3.10 -y

In [1]:
!pip install torch
!pip install pandas
!pip install numpy 

In [2]:
import torch 
import pandas as pd
import numpy as np

In [3]:
from datasets import load_dataset

dataset = load_dataset("imdb")

print(dataset["train"][0])

c:\Users\Gaurav B V\anaconda3\envs\bot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Gaurav B V\anaconda3\envs\bot\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Gaurav B V\.cache\huggingface\hub\datasets--imdb. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [4]:
dataset["train"].features

{'text': Value(dtype='string', id=None),
 'label': ClassLabel(names=['neg', 'pos'], id=None)}

Model time

In [2]:
!pip install transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 26.8 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 23.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [transformers] [transformers]


In [8]:
!pip install tqdm


In [47]:
import torch
import torch.nn as nn
from transformers import BertConfig
from tqdm.auto import tqdm

class CustomFeedForwardLayer():
    def __init__(self,config, rank = 64):

        self.linear1 = nn.Linear(config.hidden_size, config.intermediate_size)
        self.activation = nn.GELU()
        self.linear2 = nn.Linear(config.intermediate_size, config.hidden_size)

        self.A = nn.Parameter(
            torch.randn(config.output_dim, config.intermediate_size, rank)
        )
        self.A2 = nn.Parameter(
            torch.randn(config.output_dim, config.intermediate_size, rank)
        )

    def forward(self, x):
        x = self.linear1(x)
        Ax = torch.einsum("bi,oik->bok", x, self.A)
        quad = torch.sum(Ax * Ax, dim=-1)
        x = self.activation(x+quad)
        x = self.linear2(x)
        Ax = torch.einsum("bi,oik->bok", x, self.A2)
        quad = torch.sum(Ax * Ax, dim=-1)
        return x + quad
    
class BertLayer(nn.Module):
    def __init__(self,config):
        
        super().__init__()

        self.attention = nn.MultiheadAttention(
            embed_dim=config.hidden_size,
            num_heads=config.num_attention_heads,
            batch_first=True)
        
        self.cff = CustomFeedForwardLayer(config)

        self.norm1 = nn.LayerNorm(config.hidden_size)
        self.norm1 = nn.LayerNorm(config.hidden_size)

    def forward(self, x,attention_mask=None):

        attention_output = self.attention(x,x,x)
        x = self.norm1(x + attention_output)

        cffn_output = self.cff(x)
        x = self.norm2(x + cffn_output)
 
        return x
        
class BertEmbeddings(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.word_embeddings = nn.Embedding(
            config.vocab_size, config.hidden_size
        )

        self.position_embeddings = nn.Embedding(
            config.max_position_embeddings, config.hidden_size
        )

        self.layer_norm = nn.LayerNorm(config.hidden_size)

    def forward(self, input_ids):

        seq_length = input_ids.size(1)

        position_ids = torch.arange(
            seq_length, device=input_ids.device
        ).unsqueeze(0)

        word_embeddings = self.word_embeddings(input_ids)
        position_embeddings = self.position_embeddings(position_ids)

        embeddings = word_embeddings + position_embeddings
        embeddings = self.layer_norm(embeddings)

        return embeddings
    

class BertEncoder(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.layers = nn.ModuleList([
            BertLayer(config) for _ in range(config.num_hidden_layers)
        ])

    def forward(self, x, attention_mask=None):
        for layer in self.layers:
            x = layer(x, attention_mask)
        return x


class MyBertModel(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.embeddings = BertEmbeddings(config)
        self.encoder = BertEncoder(config)

    def forward(self, input_ids, attention_mask=None):

        x = self.embeddings(input_ids)
        x = self.encoder(x, attention_mask)

        return x


In [48]:
config = BertConfig(
    vocab_size=30522,
    hidden_size=256,
    num_hidden_layers=12,
    num_attention_heads=4,
    intermediate_size=1024,
    output_dim = 256
)

model = MyBertModel(config)

In [23]:
import torch
import torch.nn as nn
from torch.optim import AdamW

# 1. Device configuration (CUDA, MPS for Mac, or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)

# 2. Loss Function (CrossEntropy for Masked Language Modeling or Classification)
criterion = nn.CrossEntropyLoss()

# 3. Optimizer (Weight decay is crucial for Transformers)
optimizer = AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)

In [28]:
from datasets import load_dataset

dataset = load_dataset("imdb")

print(dataset["train"][0])

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [31]:
from transformers import BertTokenizer
from torch.utils.data import DataLoader

# Load the standard BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(examples):
    # Padding and truncation are essential for batching
    return tokenizer(
        examples["text"], 
        padding="max_length", 
        truncation=True, 
        max_length=128 # You can increase this to 256 to match your config
    )

# Apply tokenization to all splits (train, test)
tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 50000/50000 [00:04<00:00, 10761.76 examples/s]


In [36]:
tokenized_datasets["train"]

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 25000
})

In [39]:
batch

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [ ]:
for epoc in range(0,10):
    epoch_loss = 0
    for i,batch in enumerate(tokenized_datasets):
        
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        model_output = model(i)
        loss = criterion(outputs.view(-1, config.vocab_size), labels.view(-1))

        # 4. Backward pass (Calculate gradients)
        loss.backward()

        # 5. Update weights
        optimizer.step()

        # Stats
        total_loss += loss.item()
        if batch_idx % 100 == 0:
            print(f"Epoch: {epoch} | Batch: {batch_idx} | Loss: {loss.item():.4f}")
    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch} complete. Average Loss: {avg_loss:.4f}")

TypeError: string indices must be integers

In [50]:
raw_input_ids = tokenized_datasets["test"][0]["input_ids"]

# 2. Convert to Tensor, Add Batch Dim (unsqueeze), and move to Device
input_tensor = torch.tensor(raw_input_ids).unsqueeze(0)

# 3. Now run the model
output = model(input_tensor)
print(output.shape)

TypeError: unsupported operand type(s) for +: 'Tensor' and 'tuple'

In [ ]:
tokenized_datasets["test"][0]

{'text': 'I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-appreciated and misunderstood. I tried to like this, I really did, but it is to good TV sci-fi as Babylon 5 is to Star Trek (the original). Silly prosthetics, cheap cardboard sets, stilted dialogues, CG that doesn\'t match the background, and painfully one-dimensional characters cannot be overcome with a \'sci-fi\' setting. (I\'m sure there are those of you out there who think Babylon 5 is good sci-fi TV. It\'s not. It\'s clichéd and uninspiring.) While US viewers might like emotion and character development, sci-fi is a genre that does not take itself seriously (cf. Star Trek). It may treat important issues, yet not as a serious philosophy. It\'s really difficult to care about the characters here as they are not simply foolish, just missing a spark of life. Their actions and reactions are wooden and predictable, often painful to watch. The makers of Earth KNOW it\'s rubbish as 